# Modèle statistique v4 — prédiction 1X2 et marchés de buts (projet académique)

**Protocole sans fuite :** les paramètres des ratings ont été réglés sur 2006-2015. Le choix du modèle s'est fait sur 2017/18-2020/21, puis un test unique a été mené sur 2021/22-2025/26, des saisons jamais utilisées auparavant. Chaque saison est prédite par un modèle entraîné uniquement sur les saisons précédentes (walk-forward), avec une pondération temporelle de demi-vie 4 saisons.

**Architecture**
1. **Ratings** : ClubElo, pi-ratings (Constantinou & Fenton 2013) et *ratings de buts à score* GAS (Koopman & Lit 2019). Dans ces derniers, chaque équipe a une force d'attaque et une force de défense sur l'échelle log des buts, mises à jour après chaque match par le gradient de la vraisemblance de Poisson.
2. **1X2** : régression logistique multinomiale sur ΔElo, |ΔElo|, ΔQ (xG approché par les tirs cadrés), ouverture, pi, GAS et ligue.
3. **Buts** : deux régressions de Poisson (domicile et extérieur) avec correction Dixon-Coles ρ. Elles donnent une grille de scores d'où l'on tire over/under, les deux équipes marquent, cage inviolée et but en 1re mi-temps.
4. **Évaluation** : log loss (critère principal), RPS, ECE, intervalles de confiance par bootstrap apparié. Les cotes des bookmakers servent uniquement de référence externe.

**Résultats sur le test 2021-2026 (8 821 matchs)** : log loss 1X2 v3 0,9819 → v4 0,9813 (gain non significatif), ECE 0,015 → 0,011. Sur les buts, score exact −5,8 ×10⁻³ et over 2,5 −4,7 ×10⁻³ ; ces deux gains sont significatifs. Les cotes restent meilleures (0,9716), car le marché intègre des informations absentes des données : blessures, compositions, informations de dernière minute.

**Ce qui a été testé puis rejeté** : Elo-buts et pi réglés sur 2015-2020 (gain en sélection non reproduit en test, donc surapprentissage de la sélection), forces Dixon-Coles dynamiques (redondantes avec GAS, lentes), ratings GAP, LightGBM.

Exécution : *Exécution → Tout exécuter* (≈ 1-2 min). Ensuite `simulation("Équipe A", "Équipe B")`.

In [1]:
!pip -q install statsmodels

In [2]:
# ==================== SIMULATION MATHÉMATIQUE — VERSION 4 (modèle statistique consolidé) ====================
# v4 = v3 + « ratings de buts à score » (modèle GAS, Koopman & Lit 2019) : pour chaque équipe une force d'attaque et une force
#      de défense sur l'échelle log des buts, mises à jour après CHAQUE match par le gradient de la vraisemblance de Poisson :
#          log λ_dom = μ_L + h_L + att_dom − déf_ext        log λ_ext = μ_L − h_L + att_ext − déf_dom
#          att_dom += η (buts_dom − λ_dom) ;  déf_ext −= η (buts_dom − λ_dom)   (et symétriquement pour l'extérieur)
#      η = 0,0125 et rétrécissement ×0,5 lors d'un changement de division, réglés par maximum de vraisemblance sur 2006-2015 SEULEMENT.
# Protocole de sélection (aucune fuite) : réglage 2006-2015 → sélection 2017/18-2020/21 → test final intouché 2021/22-2025/26.
#   Rejeté après test : Elo-buts et pi-ratings « réglés » sur 2015-2020 (gain en sélection qui ne se reproduit pas en test),
#   forces Dixon-Coles dynamiques (redondantes avec GAS et très lentes), ratings GAP, LightGBM (moins bon que la régression).
# Conservé de v3 : ClubElo, pi-ratings, ΔQ (xG approché), ouverture, |ΔElo|, effet ligue, niveau de buts de la ligue, Dixon-Coles ρ,
#   pondération temporelle (demi-vie 4 saisons), marchés dérivés, cotes justes, méthode de Shin, indice de fiabilité.
import numpy as np, pandas as pd, difflib, warnings
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from scipy.optimize import minimize_scalar
from scipy.stats import poisson
warnings.filterwarnings("ignore")

LIGUES = {"E0": "Premier League", "SP1": "Liga", "I1": "Serie A", "D1": "Bundesliga", "F1": "Ligue 1"}
DEMI_VIE = 4                                                    # saisons
URL = "https://raw.githubusercontent.com/xgabora/Club-Football-Match-Data-2000-2025/main/data/Matches.csv"
raw = pd.read_csv(URL, parse_dates=["MatchDate"], low_memory=False)
raw["Season"] = np.where(raw.MatchDate.dt.month >= 7, raw.MatchDate.dt.year, raw.MatchDate.dt.year - 1)

# ---------- 1. Pi-ratings sur TOUTES les ligues (les promus arrivent avec une note) ----------
R_all = raw.dropna(subset=["HomeTeam", "AwayTeam"]).sort_values("MatchDate", kind="stable").reset_index(drop=True)
PI = {}
def pi_ratings(d, lr=0.035, gamma=0.7, b=10, c=3):
    out = np.zeros((len(d), 4))
    for i, (h, a, gh, ga) in enumerate(zip(d.HomeTeam.values, d.AwayTeam.values, d.FTHome.values, d.FTAway.values)):
        rh = PI.setdefault(h, [0.0, 0.0]); ra = PI.setdefault(a, [0.0, 0.0])          # [note domicile, note extérieur]
        out[i] = rh[0], rh[1], ra[0], ra[1]                                          # valeurs AVANT le match
        if np.isnan(gh): continue
        eh = np.sign(rh[0]) * (b ** (abs(rh[0]) / c) - 1); ea = np.sign(ra[1]) * (b ** (abs(ra[1]) / c) - 1)
        e = (gh - ga) - (eh - ea); s = np.sign(e) * c * np.log10(1 + abs(e))
        o0, o1 = rh[0], ra[1]
        rh[0] += s * lr; rh[1] += (rh[0] - o0) * gamma; ra[1] -= s * lr; ra[0] += (ra[1] - o1) * gamma
    return out
R_all[["pi_hh", "pi_ha", "pi_ah", "pi_aa"]] = pi_ratings(R_all)

# ---------- 1 bis. Ratings de buts à score (GAS) sur toutes les ligues ----------
GAS = {"att": {}, "def": {}, "mu": {}, "ha": {}, "last": {}}
def gas_ratings(d, eta=0.0125, eta_l=0.0001, shrink=0.5):
    att, dfn, mu, ha, last = GAS["att"], GAS["def"], GAS["mu"], GAS["ha"], GAS["last"]
    out = np.full((len(d), 2), np.nan)
    for i, (h, a, L, gh, ga) in enumerate(zip(d.HomeTeam.values, d.AwayTeam.values, d.Division.values, d.FTHome.values, d.FTAway.values)):
        m = mu.setdefault(L, np.log(1.3)); hh = ha.setdefault(L, 0.15)
        for t in (h, a):
            if t in last and last[t] != L: att[t] *= shrink; dfn[t] *= shrink      # promu / relégué
            last[t] = L
        ah, dh, aa, da = att.get(h, 0.), dfn.get(h, 0.), att.get(a, 0.), dfn.get(a, 0.)
        lh, la = m + hh + ah - da, m - hh + aa - dh
        out[i] = lh, la                                                           # valeurs AVANT le match
        if np.isnan(gh) or np.isnan(ga): continue
        eh, ea = gh - np.exp(lh), ga - np.exp(la)
        att[h], dfn[a], att[a], dfn[h] = ah + eta * eh, da - eta * eh, aa + eta * ea, dh - eta * ea
        mu[L], ha[L] = m + eta_l * (eh + ea), hh + eta_l * (eh - ea)
    return out
R_all[["g_lh", "g_la"]] = gas_ratings(R_all)

# ---------- 2. Variables d'avant-match, 5 grands championnats ----------
D = R_all[R_all.Division.isin(LIGUES) & (R_all.Season >= 2005)].dropna(subset=["FTResult"]).reset_index(drop=True)
D["mid"] = np.arange(len(D)); D["y"] = D.FTResult.map({"H": 0, "D": 1, "A": 2})
def side(home):
    s, o = ("Home", "Away") if home else ("Away", "Home")
    return pd.DataFrame({"mid": D.mid, "date": D.MatchDate, "Division": D.Division, "team": D[f"{s}Team"], "home": int(home),
                         "gf": D[f"FT{s}"], "ga": D[f"FT{o}"], "sot_f": D[f"{s}Target"], "sot_a": D[f"{o}Target"], "elo": D[f"{s}Elo"]})
L = pd.concat([side(True), side(False)]).sort_values(["team", "date", "mid"]).reset_index(drop=True)
c_L = L.groupby("Division").gf.sum() / L.groupby("Division").sot_f.sum()
L["xg_f"], L["xg_a"] = L.sot_f * L.Division.map(c_L), L.sot_a * L.Division.map(c_L)
g = L.groupby("team")
mf = g.xg_f.transform(lambda x: x.shift(1).rolling(10, min_periods=5).mean())
ma = g.xg_a.transform(lambda x: x.shift(1).rolling(10, min_periods=5).mean())
L["Q"], L["O"] = mf - ma, mf + ma                                      # qualité de jeu, ouverture
H, A = L[L.home == 1].set_index("mid"), L[L.home == 0].set_index("mid")
X = D.set_index("mid")
X["dElo"] = (X.HomeElo - X.AwayElo) / 100; X["absElo"] = X.dElo.abs()
X["g_lh"], X["g_la"] = D.set_index("mid").g_lh, D.set_index("mid").g_la     # log λ GAS avant match
X["dQ"] = H.Q - A.Q; X["Otot"] = H.O + A.O - 5.5                # centré sur la moyenne (~5,5)
X["pi_diff"] = X.pi_hh - X.pi_aa; X["pi_diff_avg"] = (X.pi_hh + X.pi_ha) / 2 - (X.pi_ah + X.pi_aa) / 2
for lg in ["SP1", "I1", "D1", "F1"]: X[f"lg_{lg}"] = (X.Division == lg).astype(float)   # Premier League = référence
FE_V3 = ["dElo", "absElo", "dQ", "Otot", "pi_diff", "pi_diff_avg", "lg_SP1", "lg_I1", "lg_D1", "lg_F1"]
FE = FE_V3 + ["g_lh", "g_la"]                                          # variables du modèle 1X2 (v4)
Dm = D.sort_values(["Division", "MatchDate", "mid"])
for c, col in (("lg_mh", "FTHome"), ("lg_ma", "FTAway")):
    Dm[c] = Dm.groupby("Division")[col].transform(lambda x: x.shift(1).rolling(380, min_periods=150).mean())
X = X.join(Dm.set_index("mid")[["lg_mh", "lg_ma"]])
X["lg_mh_c"], X["lg_ma_c"] = X.lg_mh - 1.5, X.lg_ma - 1.15            # niveau de buts de la ligue (380 derniers matchs), centré
FE_B = FE + ["lg_mh_c", "lg_ma_c"]
FE_B_V3 = FE_V3 + ["lg_mh_c", "lg_ma_c"]                                     # variables du modèle de buts
X = X.dropna(subset=FE_B)

def poids(TR, s_ref): return 0.5 ** ((s_ref - TR.Season) / DEMI_VIE)
def fit_logit(TR, s_ref, cols=None):
    cols = cols or FE
    return LogisticRegression(C=1e6, max_iter=30000, tol=1e-8).fit(TR[cols], TR.y, sample_weight=poids(TR, s_ref))
def tau(x, y, lh, la, r):
    t = np.ones(np.broadcast(x, y, lh).shape)
    t = np.where((x == 0) & (y == 0), 1 - lh * la * r, t); t = np.where((x == 0) & (y == 1), 1 + lh * r, t)
    t = np.where((x == 1) & (y == 0), 1 + la * r, t);     t = np.where((x == 1) & (y == 1), 1 - r, t)
    return t
def fit_buts(TR, s_ref, cols=None):
    cols = cols or FE_B
    w, Xa = poids(TR, s_ref), sm.add_constant(TR[cols])
    gh = sm.GLM(TR.FTHome, Xa, family=sm.families.Poisson(), freq_weights=w).fit()
    ga = sm.GLM(TR.FTAway, Xa, family=sm.families.Poisson(), freq_weights=w).fit()
    lh, la = gh.predict(Xa).values, ga.predict(Xa).values
    nll = lambda r: -np.sum(w * np.log(np.clip(tau(TR.FTHome.values, TR.FTAway.values, lh, la, r), 1e-12, None)))
    return gh, ga, minimize_scalar(nll, bounds=(-.3, .3), method="bounded").x
def grille(lh, la, rho, gmax=10):
    G = np.arange(gmax + 1)
    M = poisson.pmf(G[:, None], lh) * poisson.pmf(G[None, :], la) * tau(G[:, None], G[None, :], lh, la, rho)
    return M / M.sum()

# ---------- 3. Validation walk-forward (chaque saison prédite avec un modèle entraîné AVANT elle) ----------
def rps(P, y): c = np.cumsum(P - np.eye(3)[y], 1)[:, :2]; return (c ** 2).sum(1) / 2
def ece(P, y):
    e = 0
    for k in range(3):
        b = pd.qcut(P[:, k], 10, labels=False, duplicates="drop"); g = pd.DataFrame({"p": P[:, k], "o": y == k, "b": b}).groupby("b")
        e += (g.p.mean() - g.o.mean()).abs().mul(g.size() / len(y)).sum()
    return e / 3
def bornes(d, k=2000): b = d[np.random.default_rng(0).integers(0, len(d), (k, len(d)))].mean(1); return np.percentile(b, [2.5, 97.5])
te = X[X.Season >= 2021]; P4, P3 = np.zeros((len(te), 3)), np.zeros((len(te), 3)); O4, O3 = np.zeros(len(te)), np.zeros(len(te))
S4, S3 = np.zeros(len(te)), np.zeros(len(te)); P_grid = np.zeros((len(te), 3))
G_ = np.arange(11); TOT = G_[:, None] + G_[None, :]
for s in sorted(te.Season.unique()):
    TR, m = X[X.Season < s], (te.Season == s).values
    P4[m] = fit_logit(TR, s - 1).predict_proba(te.loc[m, FE]); P3[m] = fit_logit(TR, s - 1, FE_V3).predict_proba(te.loc[m, FE_V3])
    for cols, O, SC in ((FE_B, O4, S4), (FE_B_V3, O3, S3)):
        gh, ga, r = fit_buts(TR, s - 1, cols); Xt = sm.add_constant(te.loc[m, cols], has_constant="add")
        lh, la = gh.predict(Xt).values, ga.predict(Xt).values
        x, y_ = te.loc[m, "FTHome"].clip(upper=10).astype(int).values, te.loc[m, "FTAway"].clip(upper=10).astype(int).values
        Ms = np.stack([grille(a_, b_, r) for a_, b_ in zip(lh, la)])
        O[m] = (Ms * (TOT > 2.5)).sum((1, 2)); SC[m] = Ms[np.arange(m.sum()), x, y_]
yt = te.y.values; n = len(yt); ov = (te.FTHome + te.FTAway > 2.5).values
llv = lambda P: -np.log(P[np.arange(n), yt]); llo = lambda p: -np.log(np.where(ov, p, 1 - p))
inv = 1 / te[["OddHome", "OddDraw", "OddAway"]].values; Pm = inv / inv.sum(1, keepdims=True); ok = ~np.isnan(Pm).any(1)
po_m = (1 / te.Over25) / (1 / te.Over25 + 1 / te.Under25); oko = po_m.notna().values
base = np.bincount(yt, minlength=3) / n
print("=" * 88); print(f"VALIDATION WALK-FORWARD — TEST FINAL INTOUCHÉ 2021/22 → 2025/26 ({n} matchs) — plus bas = mieux"); print("=" * 88)
print(f"  {'1X2':28s} {'log loss':>9s} {'RPS':>7s} {'ECE':>6s} {'bon résultat':>13s}")
for nm, P in [("Fréquences (sans modèle)", np.tile(base, (n, 1))), ("Modèle v3", P3), ("Modèle v4", P4)]:
    print(f"  {nm:28s} {llv(P).mean():9.4f} {rps(P, yt).mean():7.4f} {ece(P, yt):6.3f} {np.mean(P.argmax(1) == yt):12.1%}")
print(f"  {'Cotes (référence externe)':28s} {-np.log(Pm[ok][np.arange(ok.sum()), yt[ok]]).mean():9.4f} {rps(Pm[ok], yt[ok]).mean():7.4f} {ece(Pm[ok], yt[ok]):6.3f} {np.mean(Pm[ok].argmax(1) == yt[ok]):12.1%}")
d = llv(P4) - llv(P3); lo_, hi_ = bornes(d)
print(f"  Gain v4 − v3 (1X2) : {1000 * d.mean():+.2f} ×10⁻³  (IC 95 % {1000 * lo_:+.2f} à {1000 * hi_:+.2f})  ·  skill score v4 {1 - llv(P4).mean() / llv(np.tile(base, (n, 1))).mean():.1%}")
print(f"\n  {'Buts':28s} {'LL score exact':>15s} {'LL over 2,5':>12s}")
print(f"  {'Modèle v3':28s} {-np.log(S3).mean():15.4f} {llo(O3).mean():12.4f}")
print(f"  {'Modèle v4':28s} {-np.log(S4).mean():15.4f} {llo(O4).mean():12.4f}")
print(f"  {'Cotes over/under 2,5':28s} {'—':>15s} {llo(po_m.values)[oko].mean():12.4f}   (v4 sur les mêmes matchs : {llo(O4)[oko].mean():.4f})")
d = -np.log(S4) + np.log(S3); lo_, hi_ = bornes(d); d2 = llo(O4) - llo(O3); lo2, hi2 = bornes(d2)
print(f"  Gain v4 − v3 : score exact {1000 * d.mean():+.2f} ×10⁻³ (IC {1000 * lo_:+.2f} à {1000 * hi_:+.2f}) · over 2,5 {1000 * d2.mean():+.2f} ×10⁻³ (IC {1000 * lo2:+.2f} à {1000 * hi2:+.2f})")
tab = pd.DataFrame({"Ligue": te.Division.map(LIGUES), "Saison": te.Season, "v3": llv(P3), "v4": llv(P4), "cotes": np.where(ok, -np.log(np.clip(Pm[np.arange(n), yt], 1e-9, 1)), np.nan)})
print("\n  Log loss 1X2 par ligue :"); print(tab.groupby("Ligue")[["v3", "v4", "cotes"]].mean().round(4).to_string())
print("\n  Log loss 1X2 par saison :"); print(tab.groupby("Saison")[["v3", "v4", "cotes"]].mean().round(4).to_string())
cal = pd.DataFrame({"p": O4, "o": ov}).groupby(pd.qcut(O4, 5), observed=True).mean()
print("\n  Calibration over 2,5 (v4) : " + " · ".join(f"prévu {r.p:.0%} → observé {r.o:.0%}" for r in cal.itertuples()))
P_new = P4

# ---------- 4. Modèle final, estimé sur TOUTES les saisons (poids max sur la plus récente) ----------
S_MAX = int(X.Season.max())
LOGIT = fit_logit(X, S_MAX); GH, GA, RHO = fit_buts(X, S_MAX)
_lam = GH.predict(sm.add_constant(X[FE_B])).values + GA.predict(sm.add_constant(X[FE_B])).values; _m = X.HTHome.notna().values
_htg = (X.HTHome + X.HTAway).values[_m]
K_HT = minimize_scalar(lambda k: -np.sum(_htg * np.log(k * _lam[_m]) - k * _lam[_m]), bounds=(.2, .7), method="bounded").x   # part des buts en 1re MT
# fiabilité : précision observée du modèle (validation) selon sa confiance, et prévisibilité en fin de saison
CONF = pd.DataFrame({"pmax": P_new.max(1), "ok": P_new.argmax(1) == yt})
CONF_BINS = [0, .42, .50, .61, 1]; CONF_ACC = CONF.groupby(pd.cut(CONF.pmax, CONF_BINS), observed=True).ok.mean().values
N_EQ = D.groupby(["Division", "Season"]).HomeTeam.nunique()
cls = list(LOGIT.classes_)                                              # 0 = dom, 1 = nul, 2 = ext
coef = pd.DataFrame({"log[P(1)/P(X)]": np.r_[LOGIT.intercept_[0] - LOGIT.intercept_[1], LOGIT.coef_[0] - LOGIT.coef_[1]],
                     "log[P(2)/P(X)]": np.r_[LOGIT.intercept_[2] - LOGIT.intercept_[1], LOGIT.coef_[2] - LOGIT.coef_[1]]}, index=["constante"] + FE)
coef = coef.join(pd.DataFrame({"log λ_dom": GH.params.values, "log λ_ext": GA.params.values}, index=["constante"] + FE_B), how="outer").loc[["constante"] + FE_B]
print("\n" + "=" * 80); print(f"FORMULES DU MODÈLE FINAL  (estimées sur 2005/06 → {S_MAX}/{S_MAX + 1 - 2000:02d}, demi-vie {DEMI_VIE} saisons ; ρ = {RHO:+.3f})"); print("=" * 80)
print(coef.round(3).to_string(na_rep="—"))
print("Lecture : z1 = somme(coef × variable) de la 1re colonne, z2 de la 2e ; P(X) = 1/(1+e^z1+e^z2), P(1) = e^z1·P(X), P(2) = e^z2·P(X)")
print("Variables : g_lh / g_la = log des buts attendus selon les ratings GAS ; les autres comme en v3.")
print(f"But en 1re mi-temps : λ_MT = {K_HT:.3f} × (λ_dom + λ_ext)  (≈ {K_HT:.0%} des buts avant la pause)")

# ---------- 5. Simulation d'un match ----------
def etat(team):
    t = L[L.team == team].sort_values("date")
    if t.empty: raise ValueError(f"Équipe inconnue : « {team} ». Proches : {difflib.get_close_matches(team, L.team.unique(), 5)}")
    last = t.tail(10)
    lg = t.Division.iloc[-1]; lgm = D[D.Division == lg].tail(380); s_cur = int(D.Season.max())
    joues = int(((D.HomeTeam == team) | (D.AwayTeam == team))[D.Season == s_cur].sum())
    n_eq = N_EQ.get((lg, s_cur), 20)
    return {"elo": t.elo.dropna().iloc[-1], "Q": last.xg_f.mean() - last.xg_a.mean(), "O": last.xg_f.mean() + last.xg_a.mean(),
            "pi": PI.get(team, [0.0, 0.0]), "lg": lg, "att": GAS["att"].get(team, 0.), "def": GAS["def"].get(team, 0.), "last": GAS["last"].get(team, lg), "date": t.date.iloc[-1].date(),
            "lg_mh": lgm.FTHome.mean(), "lg_ma": lgm.FTAway.mean(), "restants": 2 * (n_eq - 1) - joues}

def shin(odds):
    pi = 1 / np.asarray(odds, float); S = pi.sum()
    f = lambda z: (np.sqrt(z * z + 4 * (1 - z) * pi * pi / S) - z).sum() / (2 * (1 - z)) - 1
    z = minimize_scalar(lambda z: f(z) ** 2, bounds=(0, .4), method="bounded").x
    return (np.sqrt(z * z + 4 * (1 - z) * pi * pi / S) - z) / (2 * (1 - z)), S - 1

def simulation(dom, ext, elo_dom=None, elo_ext=None, cotes=None, n_sim=100_000, seed=0, detail=False):
    """cotes = (cote_1, cote_X, cote_2) optionnel : pour comparer le modèle au marché (analyse académique)."""
    a, b = etat(dom), etat(ext)
    ea, eb = elo_dom or a["elo"], elo_ext or b["elo"]
    v = {"dElo": (ea - eb) / 100, "dQ": a["Q"] - b["Q"], "Otot": a["O"] + b["O"] - 5.5,
         "pi_diff": a["pi"][0] - b["pi"][1], "pi_diff_avg": np.mean(a["pi"]) - np.mean(b["pi"])}
    v["absElo"] = abs(v["dElo"])
    L_ = a["lg"]; sh = lambda e, k: e[k] * (0.5 if e["last"] != L_ else 1.0)                  # promu : rétrécissement
    v["g_lh"] = GAS["mu"][L_] + GAS["ha"][L_] + sh(a, "att") - sh(b, "def")
    v["g_la"] = GAS["mu"][L_] - GAS["ha"][L_] + sh(b, "att") - sh(a, "def")
    for lg in ["SP1", "I1", "D1", "F1"]: v[f"lg_{lg}"] = float(a["lg"] == lg)
    v["lg_mh_c"], v["lg_ma_c"] = a["lg_mh"] - 1.5, a["lg_ma"] - 1.15
    xm = pd.DataFrame([v])[FE]; xb = pd.DataFrame([v])[FE_B]
    p = LOGIT.predict_proba(xm)[0]
    cl = coef.loc[["constante"] + FE]; z1 = cl["log[P(1)/P(X)]"].values @ np.r_[1, xm.values[0]]; z2 = cl["log[P(2)/P(X)]"].values @ np.r_[1, xm.values[0]]
    Xc = sm.add_constant(xb, has_constant="add"); lh, la = GH.predict(Xc).iloc[0], GA.predict(Xc).iloc[0]
    M = grille(lh, la, RHO)
    rng = np.random.default_rng(seed); idx = rng.choice(M.size, n_sim, p=M.ravel()); sx, sy = np.divmod(idx, M.shape[1]); tot = sx + sy
    # incertitude : bootstrap du modèle sur 20 rééchantillonnages (detail=False pour aller plus vite)
    if detail:
        boot = []
        for k in range(20):
            Xb = X.sample(len(X), replace=True, random_state=k); boot.append(fit_logit(Xb, S_MAX).predict_proba(xm)[0])
        lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    print("\n" + "=" * 80); print(f"SIMULATION MATHÉMATIQUE v4 : {dom} – {ext}   ({LIGUES.get(a['lg'], a['lg'])})"); print("=" * 80)
    print(f"Elo {ea:.0f} vs {eb:.0f} (ΔElo {v['dElo']:+.2f})  ·  qualité de jeu {a['Q']:+.2f} vs {b['Q']:+.2f} (ΔQ {v['dQ']:+.2f})")
    print(f"GAS : att/déf {dom} {a['att']:+.2f}/{a['def']:+.2f} · {ext} {b['att']:+.2f}/{b['def']:+.2f} → buts « bruts » {np.exp(v['g_lh']):.2f} – {np.exp(v['g_la']):.2f}")
    print(f"pi-ratings dom {a['pi'][0]:+.2f} / ext {b['pi'][1]:+.2f}  ·  ouverture attendue {v['Otot'] + 5.5:.2f}  ·  données au {max(a['date'], b['date'])}")
    print(f"z1 = {z1:+.3f}   z2 = {z2:+.3f}")
    ic = (lambda k: f"   (IC 95 % {lo[k]:.1%} – {hi[k]:.1%})") if detail else (lambda k: "")
    print(f"\n  ► Victoire {dom:<18s} {p[0]:6.1%}{ic(0)}")
    print(f"  ► Match nul {'':17s} {p[1]:6.1%}{ic(1)}")
    print(f"  ► Victoire {ext:<18s} {p[2]:6.1%}{ic(2)}")
    cs_d, cs_e, btts = M[:, 0].sum(), M[0, :].sum(), M[1:, 1:].sum()
    ht = 1 - np.exp(-K_HT * (lh + la))
    print(f"\n  Buts attendus : {dom} {lh:.2f} · {ext} {la:.2f}   ·   Monte-Carlo ({n_sim:,}) : 1 {np.mean(sx > sy):.1%} · X {np.mean(sx == sy):.1%} · 2 {np.mean(sx < sy):.1%}")
    mk = [("Victoire " + dom, p[0]), ("Match nul", p[1]), ("Victoire " + ext, p[2]),
          ("Plus de 1,5 but", np.mean(tot > 1.5)), ("Plus de 2,5 buts", np.mean(tot > 2.5)), ("Plus de 3,5 buts", np.mean(tot > 3.5)),
          ("Les deux équipes marquent", btts), ("But en 1re mi-temps", ht),
          (f"Cage inviolée {dom}", cs_d), (f"Cage inviolée {ext}", cs_e)]
    print("\n  Marché                              Probabilité   Cote juste (1/p)")
    for nm, q in mk: print(f"  {nm:35s} {q:8.1%}      {1 / q:6.2f}")
    top = sorted(((M[i, j], f"{i}-{j}") for i in range(6) for j in range(6)), reverse=True)[:5]
    print("  Scores les plus probables : " + " · ".join(f"{s} {q:.1%}" for q, s in top))
    k = int(np.digitize(p.max(), CONF_BINS[1:-1]))
    niveau = ["faible", "moyenne", "bonne", "élevée"][k]
    print(f"\n  Fiabilité du pronostic : {niveau} — quand le modèle est aussi confiant, son issue favorite s'est réalisée {CONF_ACC[k]:.0%} du temps (2021–2026)")
    if min(a["restants"], b["restants"]) <= 5:
        print("  ⚠ Fin de saison : ces matchs sont historiquement moins prévisibles (skill score 6,9 % contre 8,9 %).")
    if cotes is not None:
        pm, marge = shin(cotes)
        print(f"\n  Comparaison aux cotes {tuple(cotes)} (marge {marge:.1%}, probabilités de Shin) :")
        for nm, qm, qc in zip(["1", "X", "2"], p, pm): print(f"    {nm} : modèle {qm:.1%} · marché {qc:.1%} · écart {100 * (qm - qc):+.1f} pts")
        print("    Rappel : le marché est en moyenne plus précis que ce modèle (log loss 1X2 0,972 contre 0,982 en test) ; un écart est d'abord un signe d'information manquante.")
    return {"1": p[0], "X": p[1], "2": p[2], "buts_dom": lh, "buts_ext": la, "over25": np.mean(tot > 2.5),
            "btts": btts, "but_1re_MT": ht, "cs_dom": cs_d, "cs_ext": cs_e, "fiabilite": niveau}

simulation("Barcelona", "Real Madrid", detail=True)
simulation("Paris SG", "Marseille", cotes=(1.45, 4.60, 6.50))   # cotes fictives, pour illustrer la comparaison


VALIDATION WALK-FORWARD — TEST FINAL INTOUCHÉ 2021/22 → 2025/26 (8918 matchs) — plus bas = mieux
  1X2                           log loss     RPS    ECE  bon résultat
  Fréquences (sans modèle)        1.0733  0.2299  0.000        43.6%
  Modèle v3                       0.9819  0.1989  0.015        52.8%
  Modèle v4                       0.9813  0.1988  0.011        52.6%
  Cotes (référence externe)       0.9716  0.1956  0.015        53.7%
  Gain v4 − v3 (1X2) : -0.55 ×10⁻³  (IC 95 % -1.86 à +0.65)  ·  skill score v4 8.6%

  Buts                          LL score exact  LL over 2,5
  Modèle v3                             2.9130       0.6807
  Modèle v4                             2.9071       0.6760
  Cotes over/under 2,5                       —       0.6715   (v4 sur les mêmes matchs : 0.6760)


  Gain v4 − v3 : score exact -5.83 ×10⁻³ (IC -8.32 à -3.49) · over 2,5 -4.66 ×10⁻³ (IC -6.52 à -2.97)

  Log loss 1X2 par ligue :
                    v3      v4   cotes
Ligue                                 
Bundesliga      0.9854  0.9851  0.9754
Liga            0.9774  0.9758  0.9666
Ligue 1         0.9964  0.9954  0.9846
Premier League  0.9755  0.9744  0.9609
Serie A         0.9770  0.9782  0.9730

  Log loss 1X2 par saison :
            v3      v4   cotes
Saison                        
2021    0.9887  0.9863  0.9778
2022    0.9835  0.9879  0.9805
2023    0.9681  0.9665  0.9567
2024    0.9806  0.9781  0.9650
2025    0.9907  0.9901  0.9802
2026    0.9328  0.9296  0.9216

  Calibration over 2,5 (v4) : prévu 42% → observé 41% · prévu 48% → observé 48% · prévu 52% → observé 53% · prévu 56% → observé 59% · prévu 65% → observé 66%



FORMULES DU MODÈLE FINAL  (estimées sur 2005/06 → 2026/27, demi-vie 4 saisons ; ρ = -0.055)
             log[P(1)/P(X)]  log[P(2)/P(X)]  log λ_dom  log λ_ext
constante             0.060          -0.052      0.202      0.105
dElo                 -0.014          -0.235      0.053     -0.083
absElo                0.097           0.076     -0.006      0.005
dQ                    0.272          -0.137      0.090     -0.099
Otot                  0.036          -0.022      0.036      0.021
pi_diff               0.067          -0.368     -0.022     -0.184
pi_diff_avg          -0.148           0.363      0.032      0.219
lg_SP1               -0.018          -0.079     -0.016     -0.031
lg_I1                -0.144          -0.045     -0.064     -0.012
lg_D1                -0.128          -0.043      0.004     -0.010
lg_F1                -0.057           0.023     -0.029      0.004
g_lh                  1.075           0.129      0.559      0.092
g_la                 -0.560           0.434      


SIMULATION MATHÉMATIQUE v4 : Barcelona – Real Madrid   (Liga)
Elo 1968 vs 1931 (ΔElo +0.37)  ·  qualité de jeu +0.96 vs +0.92 (ΔQ +0.03)
GAS : att/déf Barcelona +0.91/+0.52 · Real Madrid +0.72/+0.59 → buts « bruts » 2.07 – 1.35
pi-ratings dom +1.74 / ext +1.37  ·  ouverture attendue 5.64  ·  données au 2026-08-31
z1 = +0.692   z2 = -0.034

  ► Victoire Barcelona           50.4%   (IC 95 % 49.0% – 53.0%)
  ► Match nul                    25.2%   (IC 95 % 23.5% – 26.8%)
  ► Victoire Real Madrid         24.4%   (IC 95 % 22.2% – 25.3%)

  Buts attendus : Barcelona 1.88 · Real Madrid 1.27   ·   Monte-Carlo (100,000) : 1 50.8% · X 23.4% · 2 25.8%

  Marché                              Probabilité   Cote juste (1/p)
  Victoire Barcelona                     50.4%        1.98
  Match nul                              25.2%        3.96
  Victoire Real Madrid                   24.4%        4.10
  Plus de 1,5 but                        82.9%        1.21
  Plus de 2,5 buts                       61.0

{'1': np.float64(0.5652997337277413),
 'X': np.float64(0.23264929615160426),
 '2': np.float64(0.2020509701206545),
 'buts_dom': np.float64(2.0384465259082645),
 'buts_ext': np.float64(1.1834423799727978),
 'over25': np.float64(0.62411),
 'btts': np.float64(0.6086732304887059),
 'but_1re_MT': np.float64(0.7557412945863087),
 'cs_dom': np.float64(0.3062228035665836),
 'cs_ext': np.float64(0.1302321526605898),
 'fiabilite': 'bonne'}

## Utilisation
`simulation(dom, ext, elo_dom=None, elo_ext=None, cotes=None, detail=False)` : `detail=True` ajoute l'intervalle de confiance (bootstrap) ; `cotes=(c1, cX, c2)` compare au marché. La fonction renvoie un dictionnaire de probabilités.

In [3]:
r = simulation("Arsenal", "Chelsea")
r


SIMULATION MATHÉMATIQUE v4 : Arsenal – Chelsea   (Premier League)
Elo 2018 vs 1840 (ΔElo +1.78)  ·  qualité de jeu +0.69 vs -0.42 (ΔQ +1.11)
GAS : att/déf Arsenal +0.65/+0.80 · Chelsea +0.51/+0.35 → buts « bruts » 1.97 – 0.88
pi-ratings dom +1.83 / ext +1.38  ·  ouverture attendue 4.12  ·  données au 2026-08-31
z1 = +1.237   z2 = -0.456

  ► Victoire Arsenal             67.8%
  ► Match nul                    19.7%
  ► Victoire Chelsea             12.5%

  Buts attendus : Arsenal 2.03 · Chelsea 0.85   ·   Monte-Carlo (100,000) : 1 64.3% · X 21.2% · 2 14.5%

  Marché                              Probabilité   Cote juste (1/p)
  Victoire Arsenal                       67.8%        1.47
  Match nul                              19.7%        5.08
  Victoire Chelsea                       12.5%        8.01
  Plus de 1,5 but                        78.7%        1.27
  Plus de 2,5 buts                       54.8%        1.82
  Plus de 3,5 buts                       32.4%        3.09
  Les deux éq

{'1': np.float64(0.6782318445428344),
 'X': np.float64(0.19695301802466977),
 '2': np.float64(0.12481513743249584),
 'buts_dom': np.float64(2.028026380176959),
 'buts_ext': np.float64(0.8486162314676471),
 'over25': np.float64(0.54812),
 'btts': np.float64(0.5020089460067159),
 'but_1re_MT': np.float64(0.7159165794101426),
 'cs_dom': np.float64(0.428006785492365),
 'cs_ext': np.float64(0.13159622582252162),
 'fiabilite': 'élevée'}